# AML - SAIGE-GENE SINGLE VARIANT - FIT NULL MODEL 

In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 
import pandas as pd
# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket

# GENERATE LIST OF SAMPLES FOR NULL MODEL

In [ ]:
# Copy list of samples which passed acaf quality control to workspace 
!gsutil cp {my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/acaf_threshold/EUR-like_acaf_qc.txt . 
# Copy list of samples with phenotype information to workspace 
!gsutil cp {my_bucket}/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt .

In [ ]:
# Load list of sample which passed quality control
acaf_sample_list = pd.read_csv("EUR-like_acaf_qc.txt", sep ='\t')[["IID"]]
# Load list of samples with phenotype information  
phenotype_file = pd.read_csv("EUR_phenotype_file.txt", sep ='\t')[["person_id"]]
# Filter for samples in both lists and save list 
acaf_sample_list[acaf_sample_list["IID"].isin(phenotype_file["person_id"])].to_csv("EUR_aml_phenotype_acaf_samplelist.txt", 
                                                                         sep ='\t', 
                                                                         header=None, 
                                                                         index=None)

In [ ]:
# Copy analysis cohort list to the workspace bucket 
!gsutil cp EUR_aml_phenotype_acaf_samplelist.txt {my_bucket}/data/saige_phenotype_files/aml_phenotype_files/

# COMMAND 

In [ ]:
%%writefile single_variant_saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_phenotype; do 
    echo "Processing phenotype: ${pheno}"
    
    step1_fitNULLGLMM.R \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --plinkFile="${variance_files}/single.variant.${pop}_array.forCate_vr" \
        --phenoFile="${phenotype}/${pop}_phenotype_file.txt" \
        --useSparseGRMtoFitNULL=TRUE \
        --useSparseGRMforVarRatio=TRUE \
        --IsOverwriteVarianceRatioFile=TRUE \
        --skipVarianceRatioEstimation=FALSE \
        --phenoCol="${pheno}" \
        --traitType=binary \
        --covarColList=sex,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
        --qCovarColList=sex \
        --sampleIDColinphenoFile=person_id \
        --SampleIDIncludeFile="${phenotype}/${pop}_${pheno}_acaf_samplelist.txt" \
        --outputPrefix="${output}/single_variant_${pop}_${pheno}_null_glmm" 
    done
done


In [ ]:
# Save saige_gene_step01.sh to workspace bucket 
!gsutil cp single_variant_saige_gene_step01.sh {my_bucket}/data/scripts/

In [ ]:
# Check saige_gene_step01.sh in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/single_variant_saige_gene_step01.sh 

# DSUB COMMAND 

In [ ]:
%%bash --out single_variant_saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 100 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/single_variant_saige_gene_step01.sh"
